In [1]:
!pip install -q cellxgene-census anndata scanpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 57.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 58.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 96.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 80.1 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, whi

In [3]:
import cellxgene_census
import scanpy as sc
import pandas as pd

print("Opening Census...")
with cellxgene_census.open_soma(census_version="2025-11-08") as census:
    
    # پیدا کردن dataset_id درست
    datasets = census["census_info"]["datasets"].read().concat().to_pandas()
    lupus_ds = datasets[datasets['collection_name'].str.contains('lupus|SLE|Perez', 
                                                                   case=False, na=False)]
    print(lupus_ds[['dataset_id','collection_name','dataset_title']].to_string())



Opening Census...
                                dataset_id                                                                                                                                     collection_name                                                                                                           dataset_title
529   07f14e26-ff0d-43c4-bfe3-bf1a94dc73c3                                                                                       A transcriptional cross species map of pancreatic islet cells                                                                                            mouse pancreatic islet cells
884   7e9bde96-b4d1-43d3-a5fe-dde6c4a6dc47  A pancreatic islet scRNA-seq atlas from 48 non-diabetic, prediabetic and type 2 diabetic individuals of matched demographic and ethnic backgrounds                                                             Reintegrated transcriptomes of Delta pancreatic islet cells
1372  37b21763-7f0f-41ae-9001-60bad6e2841d       

In [4]:
with cellxgene_census.open_soma(census_version="2025-11-08") as census:
    adata = cellxgene_census.get_anndata(
        census,
        organism="Homo sapiens",
        obs_value_filter="dataset_id == '218acb0f-9f2f-4f76-b90b-15a4b7c7f629'",
    )

print(f"Cells: {adata.n_obs}")
print(f"Genes: {adata.n_vars}")
print(f"Patients: {adata.obs['donor_id'].nunique()}")
print(adata.obs['disease'].value_counts())


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


Cells: 1263676
Genes: 61497
Patients: 261
disease
systemic lupus erythematosus                           777258
normal                                                 486418
lymphoid interstitial pneumonia                             0
lymphangioleiomyomatosis                                    0
lung large cell carcinoma                                   0
                                                        ...  
dementia || Alzheimer disease || Lewy body dementia         0
dementia || Alzheimer disease                               0
dementia                                                    0
cytomegalovirus infection                                   0
dementia || Alzheimer disease || diabetes mellitus          0
Name: count, Length: 259, dtype: int64


In [5]:
print(adata.obs[['donor_id','disease','self_reported_ethnicity','sex']].head(10))

# ببین disease_group یا SLEDAI وجود داره؟
for col in adata.obs.columns:
    print(col, ':', adata.obs[col].dtype, '|', adata.obs[col].nunique(), 'unique')


   donor_id                       disease self_reported_ethnicity     sex
0    HC-546                        normal                   Asian  female
1      1132  systemic lupus erythematosus       European American  female
2  FLARE006  systemic lupus erythematosus       European American  female
3      1110  systemic lupus erythematosus       European American  female
4      1479  systemic lupus erythematosus                   Asian  female
5      1334  systemic lupus erythematosus                   Asian  female
6      1333  systemic lupus erythematosus                   Asian  female
7   IGTB670                        normal       European American  female
8      1368  systemic lupus erythematosus       European American    male
9  IGTB1506                        normal       European American  female
soma_joinid : int64 | 1263676 unique
dataset_id : category | 1 unique
assay : category | 1 unique
assay_ontology_term_id : category | 1 unique
cell_type : category | 11 unique
cell_type_

In [6]:
import numpy as np

def extract_group(donor_id):
    if donor_id.startswith('FLARE'):
        return 'Flare'
    elif donor_id.startswith('HC') or donor_id.startswith('IGTB'):
        return 'Healthy'
    else:
        return 'Managed'

adata.obs['disease_group'] = adata.obs['donor_id'].astype(str).apply(extract_group)

print(adata.obs['disease_group'].value_counts())
print(f"\nUnique patients per group:")
print(adata.obs.groupby('disease_group')['donor_id'].nunique())


disease_group
Managed    720902
Healthy    484931
Flare       57843
Name: count, dtype: int64

Unique patients per group:
disease_group
Flare       14
Healthy     98
Managed    149
Name: donor_id, dtype: int64


In [7]:
# QC
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
adata = adata[adata.obs['pct_counts_mt'] < 20].copy()

print(f"After QC: {adata.n_obs} cells")

# Normalize
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)

# Patient summary
patient_summary = adata.obs.groupby('donor_id').agg(
    group=('disease_group', 'first'),
    n_cells=('disease_group', 'count'),
    cell_types=('cell_type', lambda x: x.nunique())
).reset_index()

print(patient_summary['group'].value_counts())
print(f"\nCells per patient — min:{patient_summary['n_cells'].min()} "
      f"max:{patient_summary['n_cells'].max()} "
      f"mean:{patient_summary['n_cells'].mean():.0f}")

# Save
adata.write_h5ad('/kaggle/working/lupus_phase1_processed.h5ad')
patient_summary.to_csv('/kaggle/working/patient_summary.csv', index=False)
print("Phase 1 COMPLETE ✅")


After QC: 1263438 cells


/tmp/ipykernel_58/1169746422.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  patient_summary = adata.obs.groupby('donor_id').agg(


group
Managed    149
Healthy     98
Flare       14
Name: count, dtype: int64

Cells per patient — min:454 max:13543 mean:4841
Phase 1 COMPLETE ✅
